### Outstanding Questions
* Of all apps submitted during {time frame}, how many days have they been/did they spend waiting to be approved?

In [ ]:
# Import the usual libraries and variables.
%run standard_imports.py

# Install a pip package in the current Jupyter kernel.
import sys
!{sys.executable} -m pip install --user pypika

In [ ]:
import datetime
from pypika import MySQLQuery, Table, Order, functions as fn

def create_query_apps_submitted_since(date="2017-01-01",
                                      approval_statuses=["NEW", "PENDING", "APPROVED", "PUBLISHED", "DENIED"]):
    developer_app = Table("developer_app")
    developer = Table("developer")

    q = MySQLQuery.from_(developer_app).join(
        developer
    ).on(
        developer.id == developer_app.developer_id
    ).select(
        developer.uuid,
        developer.name,
        developer_app.id,
        developer_app.uuid,
        developer_app.name,
        developer_app.first_submitted_time,
        developer_app.approval_status
    ).where(
        developer.name != "Clover"
    ).where(
        (developer_app.approval_status.isin(approval_statuses)) &
        (developer_app.deleted_time.isnull()) &
        (developer_app.first_submitted_time.notnull())
    ).where(
        developer_app.first_submitted_time >= fn.Date(date)
    ).orderby(developer_app.first_submitted_time, order=Order.asc)

    print(q)
    query = q.get_sql()
    return query
    
# Year to date
start_date = datetime.datetime.utcnow().replace(month=1, day=1).date()

pending_apps_query = create_query_apps_submitted_since(approval_statuses=["PENDING"])
db = Db("~/.clover/p801.cfg")
df = pd.read_sql(pending_apps_query, con=db.conn)
db.close()

df.head()

In [ ]:
import datetime

def days_since(x):
    current = datetime.datetime.utcnow()
    diff = current - x
    return diff.days
    
df["days_pending"] = df["first_submitted_time"].map(lambda x: days_since(x))
df.head()

In [ ]:
# Plot length pending as a histogram.

plt.hist(df['days_pending'].dropna(), color='green', edgecolor='black', bins=int(df['days_pending'].max()/7))
plt.title('Histogram of Pending Apps')
plt.xlabel('Days Pending')
plt.ylabel('Apps')

print(df["days_pending"].describe())

# print "Range of days pending: " + str(df['days_pending'].min()) + ", " + str(df['days_pending'].max())
# print "Mean (average) days pending: " + str(df['days_pending'].mean())
# print "Median days pending: " + str(df['days_pending'].median())
# print "Modal days pending: " + str(df['days_pending'].mode().values[0])

In [ ]:
print("Less than 30 Days: " + str(len(df[(df['days_pending']<30)])))
print("More than 30, Less than 60 Days: " + str(len(df[(df['days_pending']>30) & (df['days_pending']<60)])))
print("More than 60 Days: " + str(len(df[(df['days_pending']>60)])))

In [ ]:
submitted_apps_query = create_query_apps_submitted_since(date=start_date)
db = Db("~/.clover/p801.cfg")
df = pd.read_sql(submitted_apps_query, con=db.conn)
db.close()

df.head()

In [ ]:
print(df['approval_status'].describe())
print(df['approval_status'].value_counts())
df['approval_status'].value_counts(normalize=True)

In [ ]:
CountStatus = pd.value_counts(df["approval_status"].values, sort=True)
CountStatus.plot.barh(title="Status of Apps Submitted Since " + start_date.strftime("%b %-d %Y"))